In [1]:
import os

path = "../data/diabetic_data.csv"
print(f"File exists: {os.path.exists(path)}")

File exists: True


In [2]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings("ignore")

print("✓ Libraries imported")

✓ Libraries imported


In [3]:
# ── Load raw data ──────────────────────────────────────────
df = pd.read_csv("../data/diabetic_data.csv")

print(f"✓ Data loaded: {df.shape[0]:,} rows · {df.shape[1]} columns")
print(f"\n── Column names ────────────────────────────────────")
print(list(df.columns))
print(f"\n── First 3 rows ────────────────────────────────────")
df.head(3)

✓ Data loaded: 101,766 rows · 50 columns

── Column names ────────────────────────────────────
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

── First 3 rows ────────────────────────────────────


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


In [4]:
# ── Missing value analysis ─────────────────────────────────
# This dataset uses '?' to represent missing values
# We need to find and handle them

# Replace ? with NaN
df_clean = df.replace("?", np.nan)

# Count missing values
missing = pd.DataFrame({
    "missing_count": df_clean.isnull().sum(),
    "missing_pct":   round(df_clean.isnull().sum() / len(df_clean) * 100, 1)
}).sort_values("missing_pct", ascending=False)

print("── Missing values (columns with any missing) ───────")
print(missing[missing["missing_count"] > 0])

── Missing values (columns with any missing) ───────
                   missing_count  missing_pct
weight                     98569         96.9
max_glu_serum              96420         94.7
A1Cresult                  84748         83.3
medical_specialty          49949         49.1
payer_code                 40256         39.6
race                        2273          2.2
diag_3                      1423          1.4
diag_2                       358          0.4
diag_1                        21          0.0


In [5]:
# ── Drop columns with too many missing values ──────────────
# weight: ~97% missing — unusable
# payer_code: ~40% missing — not useful for readmission
# medical_specialty: ~49% missing — too many gaps

cols_to_drop = [
    "weight",
    "payer_code",
    "medical_specialty",
    "examide",           # only one unique value - no predictive power
    "citoglipton",       # only one unique value - no predictive power
]

df_clean = df_clean.drop(columns=cols_to_drop)

print(f"✓ Dropped {len(cols_to_drop)} columns")
print(f"✓ Remaining columns: {df_clean.shape[1]}")

✓ Dropped 5 columns
✓ Remaining columns: 45


In [6]:
# ── Fill remaining missing values ──────────────────────────

# race — fill missing with 'Unknown'
df_clean["race"] = df_clean["race"].fillna("Unknown")

# diag_1, diag_2, diag_3 — primary and secondary diagnoses
# fill missing with '0' meaning not recorded
df_clean["diag_1"] = df_clean["diag_1"].fillna("0")
df_clean["diag_2"] = df_clean["diag_2"].fillna("0")
df_clean["diag_3"] = df_clean["diag_3"].fillna("0")

print("✓ Missing values handled")
print(f"\nRemaining nulls: {df_clean.isnull().sum().sum()}")

✓ Missing values handled

Remaining nulls: 181168


In [7]:
# ── Create binary target variable ─────────────────────────
# Original readmitted column has 3 values:
# '<30'  = readmitted within 30 days  ← this is what we want to predict
# '>30'  = readmitted after 30 days
# 'NO'   = not readmitted

# Binary: 1 = readmitted within 30 days, 0 = everything else
df_clean["readmitted_30"] = (
    df_clean["readmitted"] == "<30").astype(int)

print("── Readmission distribution ────────────────────────")
print(df_clean["readmitted"].value_counts())
print(f"\n── Binary target ───────────────────────────────────")
print(df_clean["readmitted_30"].value_counts())
print(f"\nReadmission rate (30 day): "
      f"{df_clean['readmitted_30'].mean()*100:.1f}%")

── Readmission distribution ────────────────────────
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

── Binary target ───────────────────────────────────
readmitted_30
0    90409
1    11357
Name: count, dtype: int64

Readmission rate (30 day): 11.2%


In [8]:
# ── Create new features ────────────────────────────────────
# This is what separates a good analyst from a great one
# We create clinically meaningful features from raw data

# 1. Total medications changed — sum of all medication change flags
med_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide",
    "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide", "insulin",
    "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone",
    "metformin-pioglitazone"
]

# Count how many medications were changed (not 'No' or 'Steady')
df_clean["n_meds_changed"] = (
    df_clean[med_cols]
    .apply(lambda col: col.isin(["Up", "Down"]))
    .sum(axis=1)
)

# 2. Total procedures + lab procedures combined
df_clean["total_procedures"] = (
    df_clean["num_procedures"] +
    df_clean["num_lab_procedures"]
)

# 3. Service utilization score
df_clean["service_utilization"] = (
    df_clean["number_outpatient"] +
    df_clean["number_emergency"] +
    df_clean["number_inpatient"]
)

# 4. Age as numeric midpoint
age_map = {
    "[0-10)":   5,  "[10-20)": 15, "[20-30)": 25,
    "[30-40)": 35,  "[40-50)": 45, "[50-60)": 55,
    "[60-70)": 65,  "[70-80)": 75, "[80-90)": 85,
    "[90-100)": 95
}
df_clean["age_numeric"] = df_clean["age"].map(age_map)

# 5. High risk diagnosis flag
# ICD-9 codes for circulatory and respiratory conditions
# which are known readmission risk factors
df_clean["high_risk_diag"] = (
    df_clean["diag_1"].str.startswith(("4", "5"))
).astype(int)

print("✓ Feature engineering complete")
print(f"\nNew features created:")
print(f"  n_meds_changed:     {df_clean['n_meds_changed'].describe()['mean']:.2f} avg")
print(f"  total_procedures:   {df_clean['total_procedures'].describe()['mean']:.2f} avg")
print(f"  service_utilization:{df_clean['service_utilization'].describe()['mean']:.2f} avg")
print(f"  age_numeric:        {df_clean['age_numeric'].describe()['mean']:.1f} avg age")
print(f"  high_risk_diag:     {df_clean['high_risk_diag'].mean()*100:.1f}% of patients")

✓ Feature engineering complete

New features created:
  n_meds_changed:     0.29 avg
  total_procedures:   44.44 avg
  service_utilization:1.20 avg
  age_numeric:        66.0 avg age
  high_risk_diag:     53.2% of patients


In [9]:
# ── Select final columns ───────────────────────────────────

final_cols = [
    # Patient identifiers
    "encounter_id",
    "patient_nbr",

    # Demographics
    "race",
    "gender",
    "age",
    "age_numeric",

    # Hospital visit info
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",

    # Engineered features
    "n_meds_changed",
    "total_procedures",
    "service_utilization",
    "high_risk_diag",

    # Key medications
    "insulin",
    "diabetesMed",
    "change",

    # A1c and glucose results
    "A1Cresult",
    "max_glu_serum",

    # Admission info
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",

    # Target variable
    "readmitted",
    "readmitted_30"
]

df_final = df_clean[final_cols].copy()

# Remove duplicate patients — keep first encounter only
df_final = df_final.drop_duplicates(
    subset="patient_nbr", keep="first")

print(f"✓ Final dataset: {df_final.shape[0]:,} rows · "
      f"{df_final.shape[1]} columns")
print(f"✓ Unique patients: {df_final['patient_nbr'].nunique():,}")
print(f"✓ Readmission rate: {df_final['readmitted_30'].mean()*100:.1f}%")

✓ Final dataset: 71,518 rows · 28 columns
✓ Unique patients: 71,518
✓ Readmission rate: 8.8%


In [10]:
# ── Save to SQLite ─────────────────────────────────────────
conn = sqlite3.connect("../data/healthcare.db")
df_final.to_sql("patients", conn, if_exists="replace", index=False)
conn.close()
print("✓ Database saved to data/healthcare.db")

# ── Save to CSV ────────────────────────────────────────────
df_final.to_csv("../data/patients_clean.csv", index=False)
print("✓ CSV saved to data/patients_clean.csv")

# ── Sanity check ───────────────────────────────────────────
conn = sqlite3.connect("../data/healthcare.db")
print("\n── Table preview ───────────────────────────────────")
print(pd.read_sql("""
    SELECT COUNT(*)                         AS total_patients,
           SUM(readmitted_30)               AS readmitted_30_day,
           ROUND(SUM(readmitted_30) * 100.0
                 / COUNT(*), 1)             AS readmission_rate_pct,
           ROUND(AVG(time_in_hospital), 1)  AS avg_days_in_hospital,
           ROUND(AVG(age_numeric), 1)       AS avg_age
    FROM patients
""", conn).to_string(index=False))
conn.close()

✓ Database saved to data/healthcare.db
✓ CSV saved to data/patients_clean.csv

── Table preview ───────────────────────────────────
 total_patients  readmitted_30_day  readmission_rate_pct  avg_days_in_hospital  avg_age
          71518               6293                   8.8                   4.3     65.7
